# Milestone M3 — Eksperimen Utama (3 Seed), 4 Ablasi Anggota, Error Gallery & Grad-CAM (Topik T3-A)
Notebook ini menjalankan seluruh eksperimen terkontrol untuk membandingkan:
1. **Perbandingan Utama (2 Paradigma × 3 Seed: 42, 123, 999):** `U-Net` vs `DeepLabV3+` pada `D1 -> D2` dan `D2 -> D1`.
2. **4 Studi Ablasi Satu-Faktor (1 Per Anggota):**
   - *Attala (`230441100144`):* Strategi adaptasi warna (`Baseline` vs `ColorJitter` vs `Macenko` vs `Reinhard`)
   - *Nafaul (`240441100125`):* Fungsi loss (`CE` vs `Dice` vs `CE+Dice`)
   - *M. Rafly (`240441100086`):* Resolusi masukan & augmentasi geometrik (`128x128` vs `256x256`, `none` vs `geom`)
   - *Naufal (`240441100038`):* Encoder & strategi fine-tuning (`ResNet-34 Full` vs `MobileNetV2 Full` vs `ResNet-34 Frozen`)
3. **Galeri Kesalahan (min. 8 kasus) + Grad-CAM (4 benar + 4 salah) + Tabel Efisiensi**.


In [ ]:
# Clone repositori di Google Colab & pasang dependensi
import os, sys, subprocess

REPO_URL = "https://github.com/attaramadhani/UAS-DL-Kelompok6-T3A-WBC-DomainShift.git"
REPO_NAME = "UAS-DL-Kelompok6-T3A-WBC-DomainShift"

if "google.colab" in sys.modules and not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(REPO_NAME):
    os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Persiapan Dataset, Split Tetap & Referensi Normalisasi dari Set Latih


In [ ]:
import csv, os
import pandas as pd
import numpy as np
from IPython.display import display
from src.dataset import download_and_extract_zheng, generate_fixed_splits
from src.stain_norm import MacenkoNormalizer, ReinhardNormalizer, select_representative_reference
from src.trainer import run_experiment
from src.xai_and_errors import save_error_gallery, generate_gradcam_grid

domain_dirs = download_and_extract_zheng("data")
split_csvs = generate_fixed_splits(domain_dirs, "splits", seed=42)

def get_train_paths(dkey):
    with open(split_csvs[dkey], encoding="utf-8") as f:
        return [os.path.join(domain_dirs[dkey], r["image_file"]) for r in csv.DictReader(f) if r["split"] == "train"]

ref_d1 = select_representative_reference(get_train_paths("D1"))
macenko_d2_to_d1 = MacenkoNormalizer().fit(ref_d1)
reinhard_d2_to_d1 = ReinhardNormalizer().fit(ref_d1)

LOG_PATH = "logs/m3_experiment_runs.csv"
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)
print("Siap menjalankan seluruh eksperimen pada split seed=42!")


## 2. Perbandingan Utama (U-Net vs DeepLabV3+ × 3 Random Seeds: 42, 123, 999)


In [ ]:
SEEDS = [42, 123, 999]
EPOCHS = 40
main_results = []

for train_dom, target_dom in [("D1", "D2"), ("D2", "D1")]:
    for arch in ["unet", "deeplabv3plus"]:
        for seed in SEEDS:
            name = f"Main_{arch}_{train_dom}_to_{target_dom}_s{seed}"
            res = run_experiment(
                run_name=name,
                owner="Tim Utama (K3/K4)",
                domain_dirs=domain_dirs,
                split_csvs=split_csvs,
                train_domain=train_dom,
                target_domain=target_dom,
                arch=arch,
                encoder_name="resnet34",
                loss_name="ce_dice",
                aug_mode="geom",
                seed=seed,
                epochs=EPOCHS,
                log_csv_path=LOG_PATH,
                save_weights_path=f"weights/{name}.pth" if seed == 42 else None,
            )
            main_results.append(res)

df_runs = pd.read_csv(LOG_PATH)
summary_main = df_runs[df_runs["run_name"].str.startswith("Main_")].groupby(
    ["train_domain", "target_domain", "arch"]
)[["in_miou_fg", "cross_dice_nuc", "cross_dice_cyto", "cross_miou_fg", "cross_hd95_nuc"]].agg(["mean", "std"])
display(summary_main)


## 3. Empat Studi Ablasi Satu-Faktor (1 Per Anggota Kelompok 6)


In [ ]:
# --- ABLASI 1: Attala (Strategi Adaptasi Warna pada U-Net D1 -> D2) ---
ablation_color_configs = [
    ("Abl_Attala_NoAdapt", "geom", None, None),
    ("Abl_Attala_ColorJitter", "color_geom", None, None),
    ("Abl_Attala_Macenko_EvalToTrain", "geom", None, macenko_d2_to_d1),
    ("Abl_Attala_Reinhard_EvalToTrain", "geom", None, reinhard_d2_to_d1),
]
attala_runs = {}
for rname, amode, tr_norm, ev_norm in ablation_color_configs:
    attala_runs[rname] = run_experiment(
        run_name=rname, owner="Attala (230441100144)",
        domain_dirs=domain_dirs, split_csvs=split_csvs,
        train_domain="D1", target_domain="D2", arch="unet",
        aug_mode=amode, train_normalizer=tr_norm, eval_normalizer=ev_norm,
        seed=42, epochs=EPOCHS, log_csv_path=LOG_PATH,
    )

# --- ABLASI 2: Nafaul (Fungsi Loss pada U-Net D1 -> D2) ---
for lname in ["ce", "dice", "ce_dice"]:
    run_experiment(
        run_name=f"Abl_Nafaul_Loss_{lname}", owner="Nafaul (240441100125)",
        domain_dirs=domain_dirs, split_csvs=split_csvs,
        train_domain="D1", target_domain="D2", arch="unet",
        loss_name=lname, seed=42, epochs=EPOCHS, log_csv_path=LOG_PATH,
    )

# --- ABLASI 3: M. Rafly (Resolusi Masukan & Augmentasi Geometrik) ---
for sz, amode in [(128, "geom"), (256, "none"), (256, "geom")]:
    run_experiment(
        run_name=f"Abl_Rafly_Sz{sz}_{amode}", owner="M. Rafly (240441100086)",
        domain_dirs=domain_dirs, split_csvs=split_csvs,
        train_domain="D1", target_domain="D2", arch="deeplabv3plus",
        input_size=sz, aug_mode=amode, seed=42, epochs=EPOCHS, log_csv_path=LOG_PATH,
    )

# --- ABLASI 4: Naufal (Encoder & Strategi Fine-Tuning) ---
for enc, frz in [("resnet34", False), ("mobilenet_v2", False), ("resnet34", True)]:
    tag = "Frozen" if frz else "Full"
    run_experiment(
        run_name=f"Abl_Naufal_{enc}_{tag}", owner="Naufal (240441100038)",
        domain_dirs=domain_dirs, split_csvs=split_csvs,
        train_domain="D1", target_domain="D2", arch="deeplabv3plus",
        encoder_name=enc, freeze_encoder=frz, seed=42, epochs=EPOCHS, log_csv_path=LOG_PATH,
    )

df_all = pd.read_csv(LOG_PATH)
print(f"Total waktu GPU seluruh eksperimen: {df_all['duration_sec'].sum()/60.0:.2f} menit")
display(df_all)


## 4. Galeri Kesalahan (Min. 8 Kasus) & Visualisasi Grad-CAM (4 Benar + 4 Salah)


In [ ]:
best_eval = attala_runs["Abl_Attala_Macenko_EvalToTrain"]
records = best_eval["cross_records"]
for r in records:
    r["domain"] = "D2 Cross-Domain"

os.makedirs("docs/figures", exist_ok=True)
err_counts = save_error_gallery(records, "docs/figures/fig4_error_gallery.png", top_k=8)
print("Frekuensi Kategori Kesalahan:", err_counts)

sorted_rec = sorted(records, key=lambda r: r["dice_fg_mean"])
worst_4 = [{**r, "tag": "Salah/Buruk"} for r in sorted_rec[:4]]
best_4 = [{**r, "tag": "Benar/Baik"} for r in sorted_rec[-4:]]
model = best_eval["model"]
target_layer = model.encoder.layer4[-1]
generate_gradcam_grid(model, target_layer, best_4 + worst_4, "docs/figures/fig5_gradcam_grid.png", target_class=2)
